<a href="https://colab.research.google.com/github/codeKrantz/peptide-binding-analysis/blob/main/CritiCL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q pandas numpy joblib xgboost openpyxl huggingface_hub esm

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import joblib
import torch

from huggingface_hub import hf_hub_download
from google.colab import files

In [ ]:
# =========================
# USER SETTINGS
# =========================
HF_REPO_ID = "KarunaAnna/CritiCL"
MODEL_FILENAME = "model_XGB.joblib"          # required
LABEL_ENCODER_FILENAME = "label_encoder.joblib"  # optional; set to None if not available
HF_REPO_TYPE = "model"                       # "model" or "dataset"
HF_REVISION = None                           # e.g. "main" or a commit hash


In [ ]:
model_path = hf_hub_download(
    repo_id=HF_REPO_ID,
    filename=MODEL_FILENAME,
    repo_type=HF_REPO_TYPE,
    revision=HF_REVISION,
)

print("Downloaded model to:", model_path)

model = joblib.load(model_path)
print("Loaded model type:", type(model))
print("Model expects n_features =", model.n_features_in_)

In [ ]:
label_encoder = None

if LABEL_ENCODER_FILENAME is not None:
    try:
        label_encoder_path = hf_hub_download(
            repo_id=HF_REPO_ID,
            filename=LABEL_ENCODER_FILENAME,
            repo_type=HF_REPO_TYPE,
            revision=HF_REVISION,
        )
        label_encoder = joblib.load(label_encoder_path)
        print("Loaded label encoder classes:", list(label_encoder.classes_))
    except Exception as e:
        print("Could not load label encoder.")
        print("Reason:", e)

In [ ]:
def load_esmc(model_name: str = "esmc_300m", device: str = None):
    from esm.models.esmc import ESMC

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    model = ESMC.from_pretrained(model_name).to(device)
    model.eval()
    return model, device


model_esmc, device = load_esmc(model_name="esmc_300m")
print("ESM-C loaded on:", device)

In [ ]:
@torch.no_grad()
def esmc_token_embeddings_aligned(model, sequence: str, device: str):
    """
    Mirrors your uploaded embedding script.
    Returns aligned per-residue embeddings of shape (L, d).
    """
    from esm.sdk.api import ESMProtein, LogitsConfig

    sequence = str(sequence).strip().upper()
    L = len(sequence)

    protein = ESMProtein(sequence=sequence)
    protein_tensor = model.encode(protein).to(device)

    out = model.logits(protein_tensor, LogitsConfig(sequence=True, return_embeddings=True))
    emb = out.embeddings

    if isinstance(emb, np.ndarray):
        emb = torch.from_numpy(emb)
    emb = emb.to(device).detach()

    if emb.dim() == 3:
        if emb.shape[0] != 1:
            raise RuntimeError(f"Unexpected batch dim: {tuple(emb.shape)}")
        emb = emb.squeeze(0)

    if emb.dim() != 2:
        raise RuntimeError(f"Expected 2D embeddings (T,d), got {tuple(emb.shape)}")

    T = emb.shape[0]
    if T == L:
        return emb
    if T == L + 2:
        return emb[1:-1, :]
    if T == L + 1:
        return emb[1:, :]
    if T > L:
        start = (T - L) // 2
        return emb[start:start + L, :]

    raise RuntimeError(f"Cannot align embeddings: tokens={T}, seq_len={L}, emb_shape={tuple(emb.shape)}")


def mean_pool_residue_embeddings(emb_Ld):
    return emb_Ld.mean(dim=0)


def clean_sequence(seq):
    if pd.isna(seq):
        return ""
    return str(seq).strip().replace(" ", "").replace("\n", "").upper()


@torch.no_grad()
def sequence_to_embedding(seq, model, device):
    seq = clean_sequence(seq)
    if not seq:
        raise ValueError("Empty sequence provided.")

    emb_Ld = esmc_token_embeddings_aligned(model, seq, device=device)
    emb_d = mean_pool_residue_embeddings(emb_Ld)

    emb = emb_d.detach().cpu().numpy().astype(np.float32, copy=False)
    return emb

In [ ]:
test_seq = "AKWYFGLICCKLQLK"
test_emb = sequence_to_embedding(test_seq, model_esmc, device)

print("Generated embedding dim:", test_emb.shape[0])
print("Model expected dim:", model.n_features_in_)

if test_emb.shape[0] != model.n_features_in_:
    raise ValueError(
        f"Embedding dimension mismatch: generated {test_emb.shape[0]}, "
        f"but model expects {model.n_features_in_}."
    )

In [ ]:
def predict_from_dataframe(df, seq_col="Sequence", cycl_col="CyclizationPattern"):
    if seq_col not in df.columns:
        raise ValueError(f"Missing required sequence column: {seq_col}")

    work = df.copy()

    if cycl_col not in work.columns:
        work[cycl_col] = ""

    embeddings = []
    for seq in work[seq_col]:
        emb = sequence_to_embedding(seq, model_esmc, device)
        embeddings.append(emb)

    X = np.vstack(embeddings)

    if X.shape[1] != model.n_features_in_:
        raise ValueError(
            f"Feature shape mismatch, expected: {model.n_features_in_}, got {X.shape[1]}"
        )

    y_pred = model.predict(X)

    out = work.reset_index(drop=True).copy()

    if label_encoder is not None:
        try:
            out["prediction"] = label_encoder.inverse_transform(np.asarray(y_pred, dtype=int))
        except Exception:
            out["prediction"] = y_pred
    else:
        out["prediction"] = y_pred

    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X)
        out["confidence_max"] = proba.max(axis=1)

        if label_encoder is not None:
            class_names = list(label_encoder.classes_)
        elif hasattr(model, "classes_"):
            class_names = [str(c) for c in model.classes_]
        else:
            class_names = [f"class_{i}" for i in range(proba.shape[1])]

        for i, cname in enumerate(class_names):
            out[f"proba_{cname}"] = proba[:, i]

    return out

In [ ]:
def run_single_sequence():
    seq = input("Enter sequence: ").strip()
    cyc = input("Enter cyclization pattern (blank allowed, metadata only): ").strip()

    df = pd.DataFrame([{
        "Sequence": seq,
        "CyclizationPattern": cyc
    }])

    result = predict_from_dataframe(df)
    return result

single_result = run_single_sequence()
single_result

In [ ]:
def run_multiple_sequences():
    n = int(input("How many sequences do you want to enter? ").strip())
    rows = []

    for i in range(n):
        print(f"\nSequence {i+1}")
        seq = input("  Enter sequence: ").strip()
        cyc = input("  Enter cyclization pattern (blank allowed, metadata only): ").strip()

        rows.append({
            "Sequence": seq,
            "CyclizationPattern": cyc
        })

    df = pd.DataFrame(rows)
    result = predict_from_dataframe(df)
    return result

multi_result = run_multiple_sequences()
multi_result

In [ ]:
def run_multiple_sequences():
    n = int(input("How many sequences do you want to enter? ").strip())
    rows = []

    for i in range(n):
        print(f"\nSequence {i+1}")
        seq = input("  Enter sequence: ").strip()
        cyc = input("  Enter cyclization pattern (blank allowed, metadata only): ").strip()

        rows.append({
            "Sequence": seq,
            "CyclizationPattern": cyc
        })

    df = pd.DataFrame(rows)
    result = predict_from_dataframe(df)
    return result

multi_result = run_multiple_sequences()
multi_result

In [ ]:
def run_uploaded_file():
    uploaded = files.upload()
    in_file = list(uploaded.keys())[0]

    if in_file.lower().endswith(".csv"):
        df = pd.read_csv(in_file, keep_default_na=False, na_values=[])
    elif in_file.lower().endswith((".xlsx", ".xls")):
        df = pd.read_excel(in_file)
    else:
        raise ValueError("Please upload a CSV or Excel file.")

    print("Detected columns:", list(df.columns))
    seq_col = input("Enter sequence column name [default: Sequence]: ").strip() or "Sequence"
    cycl_col = input("Enter cyclization column name [default: CyclizationPattern]: ").strip() or "CyclizationPattern"

    total_sequences = len(df)
    processed_sequences = 0
    results_list = []
    chunk_size = 100

    if total_sequences == 0:
        print("No sequences to process.")
        return pd.DataFrame()

    print(f"Processing {total_sequences} sequences...")

    for i in range(0, total_sequences, chunk_size):
        chunk_df = df.iloc[i:i + chunk_size].copy() # Use .copy() to avoid SettingWithCopyWarning

        # Call the existing predict_from_dataframe function for each chunk
        chunk_result = predict_from_dataframe(chunk_df, seq_col=seq_col, cycl_col=cycl_col)
        results_list.append(chunk_result)

        processed_sequences += len(chunk_df)
        remaining_sequences = total_sequences - processed_sequences
        print(f"Processed {processed_sequences} sequences. {remaining_sequences} left.")

    # Concatenate all chunk results
    result = pd.concat(results_list, ignore_index=True)

    return result

upload_result = run_uploaded_file()
upload_result.head()

In [ ]:
def save_results(df, default_name="xgb_predictions.csv"):
    out_name = input(f"Output filename [default: {default_name}]: ").strip() or default_name

    if out_name.lower().endswith(".csv"):
        df.to_csv(out_name, index=False)
    elif out_name.lower().endswith((".xlsx", ".xls")):
        df.to_excel(out_name, index=False)
    else:
        out_name += ".csv"
        df.to_csv(out_name, index=False)

    print("Saved:", out_name)
    files.download(out_name)

# Example:
# save_results(single_result)
# save_results(multi_result)
save_results(upload_result)

In [ ]:
def run_menu():
    print("\nChoose input mode:")
    print("1 = Single sequence")
    print("2 = Multiple sequences manually")
    print("3 = Upload CSV/Excel")

    choice = input("Enter 1, 2, or 3: ").strip()

    if choice == "1":
        res = run_single_sequence()
    elif choice == "2":
        res = run_multiple_sequences()
    elif choice == "3":
        res = run_uploaded_file()
    else:
        raise ValueError("Invalid choice. Please enter 1, 2, or 3.")

    display(res)
    return res

results_df = run_menu()